# Controller Scoreboard: Frozen Results Made Visual (METR-01/02)

Grouped bars per metric plus the holdout cost ranking, read ONLY from
`results/comparison.csv` (frozen 14-column contract, 8 rows: 4 controllers
x 2 weathers). No controller re-runs, no metric recomputation — the CSV
is the verdict. Holdout block (seed-21) carries the METR-02 verdict:
GA fuzzy below on/off AND below PID.



In [ ]:
import csv
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.metrics import COLUMNS  # frozen column contract, never retyped
from src.weather_profile_generator import generate_weather_profile

print("columns:", len(COLUMNS))


In [ ]:
# Weather input: regenerate the seed-7 profile only if absent, then load.
# (Pattern-uniformity guard — 05 needs no simulation weather; the CSV is
# the only data source. Kept verbatim so the mechanical audit is uniform.)
CSV = "weather/weather_profile.csv"
if not os.path.exists(CSV):
    generate_weather_profile(CSV, seed=7)



In [ ]:
# Scoreboard load: stdlib csv only (tabular library forbidden by D-02).
# Fail-closed provenance: header must equal COLUMNS, 8 rows expected.
CSV_PATH = "results/comparison.csv"
with open(CSV_PATH) as f:
    rows = list(csv.DictReader(f))
assert list(rows[0].keys()) == COLUMNS, "stale artifact: header != COLUMNS"
assert len(rows) == 8, "expected 4 controllers x 2 weathers"
print("loaded %d rows x %d cols from %s" % (len(rows), len(COLUMNS), CSV_PATH))
for r in rows:
    print("%-8s %-13s cost %8.2f rank %s" % (r["controller"], r["weather"], float(r["cost"]), r["rank"]))



In [ ]:
# Grouped bars per metric (both weathers) + holdout ranking + settling note.
# Display order onoff, pid, fuzzy, ga_fuzzy; cost panel is log-scale (6x
# range would crush GA on linear); settling_min is inf everywhere so it is
# a text annotation, never bars. Rank column read directly, never recomputed.
ORDER = ["onoff", "pid", "fuzzy", "ga_fuzzy"]
DISPLAY = {"onoff": "on/off", "pid": "PID", "fuzzy": "hand fuzzy", "ga_fuzzy": "GA fuzzy"}
WEATHERS = ["train-seed7", "holdout-seed21"]
val = {(r["controller"], r["weather"], m): float(r[m])
       for r in rows for m in ("iae_T", "iae_H", "energy", "cost")}
holdout = [r for r in rows if r["weather"] == "holdout-seed21"]
# Ranking follows the CSV rank column; stable sort keeps CSV row order as tiebreak.
ranked = sorted(holdout, key=lambda r: int(r["rank"]))

fig, ax = plt.subplots(3, 2, figsize=(12, 12))
x = np.arange(len(ORDER))
w = 0.35
for j, metric in enumerate(["iae_T", "iae_H", "energy", "cost"]):
    a = ax[j // 2, j % 2]
    a.bar(x - w / 2, [val[(c, WEATHERS[0], metric)] for c in ORDER], w, label="train-seed7")
    a.bar(x + w / 2, [val[(c, WEATHERS[1], metric)] for c in ORDER], w, label="holdout-seed21")
    a.set_xticks(x)
    a.set_xticklabels([DISPLAY[c] for c in ORDER])
    a.set_title(metric)
    a.legend(fontsize=8)
ax[1, 1].set_yscale("log")
ax[1, 1].set_title("cost (log scale)")
ra = ax[2, 0]
ra.barh([DISPLAY[r["controller"]] for r in ranked], [float(r["cost"]) for r in ranked])
ra.set_xlabel("holdout cost")
ra.set_title("Holdout cost ranking (CSV rank column)")
for i, r in enumerate(ranked):
    ra.text(float(r["cost"]), i, " #%s" % r["rank"], va="center", fontsize=9)
sa = ax[2, 1]
sa.axis("off")
sa.text(0.05, 0.6, "settling: all controllers never settled\n(joint \u00b10.5 \u00b0C / \u00b13 %RH band)", fontsize=11)
sa.text(0.05, 0.3, "overshoot_H saturates at 30.0 for on/off, PID, fuzzy\n(structural note, not a GA defect)", fontsize=11)



In [ ]:
# Holdout verdict: RELATIVE ordering only — GA below on/off AND below PID.
# GA about 114.28, on/off about 513.75, PID about 645.26 (comments only).
costs = {r["controller"]: float(r["cost"]) for r in holdout}
assert costs["ga_fuzzy"] < costs["onoff"], costs
assert costs["ga_fuzzy"] < costs["pid"], costs
assert ranked[0]["controller"] == "ga_fuzzy", [r["controller"] for r in ranked]
print("holdout: GA %.2f < on/off %.2f AND GA < PID %.2f" % (costs["ga_fuzzy"], costs["onoff"], costs["pid"]))
print("verdict: GA < on/off AND GA < PID on holdout \u2014 METR-02 HOLDS")



In [ ]:
fig.suptitle("Controller scoreboard from results/comparison.csv (frozen)")
fig.tight_layout()
fig.savefig("notebooks/05_comparison.png", dpi=120)
plt.show()
print("saved notebooks/05_comparison.png")

